# Build shared 5000-user split for all recommender notebooks

This notebook creates one shared experimental split and saves it to disk so every recommender notebook and the bandit notebook can reuse exactly the same users and the same train/test split.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
POSITIVE_THRESHOLD = 4.0
MIN_POSITIVE_INTERACTIONS = 5
MAX_POSITIVE_INTERACTIONS = 250
MAX_USERS = 5000
TEST_FRACTION = 0.2

PROJECT_ROOT = Path.cwd().resolve()
for c in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (c / 'dataset').exists():
        PROJECT_ROOT = c
        break

DATASET_DIR = PROJECT_ROOT / 'dataset'
SPLIT_DIR = PROJECT_ROOT / 'data' / 'results' / 'shared_split'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SPLIT_DIR:', SPLIT_DIR)


PROJECT_ROOT: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system
SPLIT_DIR: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system/data/results/shared_split


In [2]:
ratings = pd.read_csv(
    DATASET_DIR / 'XWines_Full_21M_ratings.csv',
    usecols=['UserID', 'WineID', 'Rating'],
    dtype={'UserID': 'int32', 'WineID': 'int32', 'Rating': 'float32'}
)
ratings = ratings.drop_duplicates(subset=['UserID', 'WineID']).copy()

positive = ratings[ratings['Rating'] >= POSITIVE_THRESHOLD].copy()
user_counts = positive['UserID'].value_counts()
eligible_users = user_counts[
    (user_counts >= MIN_POSITIVE_INTERACTIONS) &
    (user_counts <= MAX_POSITIVE_INTERACTIONS)
].index
positive = positive[positive['UserID'].isin(eligible_users)].copy()

rng = np.random.default_rng(RANDOM_STATE)
sampled_users = rng.choice(
    positive['UserID'].unique(),
    size=min(MAX_USERS, positive['UserID'].nunique()),
    replace=False
)
positive = positive[positive['UserID'].isin(sampled_users)].copy()

print('eligible positive users:', len(eligible_users))
print('sampled users:', len(sampled_users))
print('positive rows after sampling:', positive.shape)


eligible positive users: 827957
sampled users: 5000
positive rows after sampling: (73319, 3)


In [3]:
def split_train_test_per_user(df, test_fraction=0.2, random_state=42):
    train_parts, test_parts = [], []
    for _, g in df.groupby('UserID'):
        g = g.sample(frac=1.0, random_state=random_state)
        n_test = max(1, int(np.ceil(len(g) * test_fraction)))
        test_g = g.iloc[:n_test]
        train_g = g.iloc[n_test:]
        if len(train_g) == 0:
            continue
        train_parts.append(train_g)
        test_parts.append(test_g)
    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)


def assign_fold_ids_per_user(df, n_folds=3, random_state=42):
    chunks = []
    for _, g in df.groupby('UserID'):
        g = g.sample(frac=1.0, random_state=random_state).copy()
        g['fold_id'] = np.arange(len(g)) % n_folds
        chunks.append(g)
    return pd.concat(chunks, ignore_index=True)


train_pos, test_pos = split_train_test_per_user(
    positive,
    test_fraction=TEST_FRACTION,
    random_state=RANDOM_STATE
)
trainval_folds = assign_fold_ids_per_user(train_pos, n_folds=3, random_state=RANDOM_STATE)
test_relevant = test_pos.groupby('UserID')['WineID'].apply(set).to_dict()

print('train_pos:', train_pos.shape)
print('test_pos:', test_pos.shape)
print('train users:', train_pos['UserID'].nunique())
print('test users:', test_pos['UserID'].nunique())


train_pos: (56673, 3)
test_pos: (16646, 3)
train users: 5000
test users: 5000


In [4]:
pd.DataFrame({'UserID': np.array(sorted(sampled_users), dtype=np.int32)}).to_csv(
    SPLIT_DIR / 'sampled_users_5000.csv', index=False
)
train_pos.to_csv(SPLIT_DIR / 'train_pos.csv', index=False)
test_pos.to_csv(SPLIT_DIR / 'test_pos.csv', index=False)
trainval_folds.to_csv(SPLIT_DIR / 'trainval_folds.csv', index=False)

test_relevant_df = (
    test_pos.groupby('UserID')['WineID']
    .apply(list)
    .reset_index(name='RelevantWineIDs')
)
test_relevant_df['RelevantWineIDs'] = test_relevant_df['RelevantWineIDs'].apply(
    lambda x: ' '.join(str(int(v)) for v in x)
)
test_relevant_df.to_csv(SPLIT_DIR / 'test_relevant.csv', index=False)

meta = pd.DataFrame([{
    'random_state': RANDOM_STATE,
    'positive_threshold': POSITIVE_THRESHOLD,
    'min_positive_interactions': MIN_POSITIVE_INTERACTIONS,
    'max_positive_interactions': MAX_POSITIVE_INTERACTIONS,
    'max_users': MAX_USERS,
    'test_fraction': TEST_FRACTION,
}])
meta.to_csv(SPLIT_DIR / 'split_meta.csv', index=False)

print('Saved files:')
for p in sorted(SPLIT_DIR.glob('*')):
    print(' -', p.name)


Saved files:
 - sampled_users_5000.csv
 - split_meta.csv
 - test_pos.csv
 - test_relevant.csv
 - train_pos.csv
 - trainval_folds.csv


## How to use these files in the other notebooks

Load `train_pos.csv`, `test_pos.csv`, `trainval_folds.csv`, and `sampled_users_5000.csv` from `data/results/shared_split/`.

Do not resample users or rebuild train/test splits inside the recommender notebooks.
